# 9.2 - Data Preparation

**Module 9 - Fine-Tuning** | Netsetos GenAI Engineering

This notebook works through Data Preparation hands-on: The three data formats; Chat templates - the silent killer; Quality filtering - drop garbage before training; Quality crushes quantity - the AlpaGasus filter; Synthetic data generation - and always filter it; The cleaning pipeline - cheapest filters first.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

One housekeeping line before the demos. The notebook leans on small deterministic examples rather than live randomness, so there is no seeding call to run here - just a note that the numbers you see are reproducible run to run.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A single comment cell that documents the reproducibility posture of the notebook; there is no executable code and nothing to import yet.

**How the code works - step by step**
- The cell holds only a comment noting that the lesson uses seeded/fixed values throughout, so outputs are stable.
- No imports, no function calls - later cells pull in `json`, `hashlib`, `re`, `unicodedata` and friends as they need them.

**In one line:** a marker that every result below is deterministic.

**What you'll see:** (no output - environment prep)

## 1 - The three data formats

Before you clean or filter anything you have to shape it, and fine-tuning data comes in exactly three shapes chosen by your goal: **instruction** (system + user + assistant) teaches task-following, **conversation** (many alternating turns) teaches multi-turn dialogue, and **preference** (prompt + chosen + rejected) teaches which answer is better - the input to DPO. All three end up as JSONL, the universal on-disk format.

In [ ]:
# THE THREE DATA FORMATS - each maps to a different fine-tuning goal.
import json

instruction = {"messages": [
    {"role": "system", "content": "You are a Netsetos support assistant."},
    {"role": "user", "content": "What is the refund policy?"},
    {"role": "assistant", "content": "Full refund within 7 days, 50% from day 8-30, none after 30."}]}

conversation = {"messages": [
    {"role": "user", "content": "What courses do you offer?"},
    {"role": "assistant", "content": "GenAI, Data Science, Cloud, and Full-Stack."},
    {"role": "user", "content": "How much is the GenAI one?"},
    {"role": "assistant", "content": "It is 14,999 rupees with lifetime access."}]}

preference = {"prompt": "What is the refund policy?",
              "chosen": "Full refund within 7 days, 50% from day 8-30, none after 30. Email support@netsetos.com.",
              "rejected": "We have a refund policy. Check the website."}

def to_jsonl(items):
    # JSONL = one JSON object per line: streamable, line-filterable, validated per record.
    return "\n".join(json.dumps(x, ensure_ascii=False) for x in items)

print("INSTRUCTION (system+user+assistant) - one JSONL line:")
print(" ", to_jsonl([instruction])[:96], "...")
print(f"CONVERSATION: {len(conversation['messages'])} alternating messages (multi-turn)")
print(f"PREFERENCE (DPO): keys = {list(preference)}  (prompt + chosen + rejected)")

# Output:
# INSTRUCTION (system+user+assistant) - one JSONL line:
#   {"messages": [{"role": "system", "content": "You are a Netsetos support assistant."}, {"role": " ...
# CONVERSATION: 4 alternating messages (multi-turn)
# PREFERENCE (DPO): keys = ['prompt', 'chosen', 'rejected']  (prompt + chosen + rejected)

**Explanation**

This is a data-shaping cell, not a model call: it hand-builds one example of each format as a plain Python dict and serializes them. The point is to see that instruction and conversation both live in a `messages` array while preference uses `prompt/chosen/rejected`, and that all three collapse to one JSON object per line.

**How the code works - step by step**
- **`instruction`** - a dict with a `messages` list carrying one system, one user, and one assistant turn (single-shot task following).
- **`conversation`** - a `messages` list of four alternating user/assistant turns (multi-turn dialogue).
- **`preference`** - a flat dict with `prompt`, `chosen`, and `rejected` keys (the A/B pair DPO trains on).
- **`to_jsonl`** - joins each record as one `json.dumps(...)` line with `ensure_ascii=False`, which is literally what "JSONL" means.
- The prints show the instruction record truncated to one line, the conversation's message count, and the preference dict's keys.

**In one line:** pick the shape by the skill you're teaching, then store it as one JSON record per line.

**What you'll see:** Three lines: the instruction record as a single truncated JSONL string, `CONVERSATION: 4 alternating messages (multi-turn)`, and `PREFERENCE (DPO): keys = ['prompt', 'chosen', 'rejected']`.

## 2 - Chat templates - the silent killer

This is the cold-open bug and the most important idea in the lesson. Your clean `messages` array is not what the model trains on - the model trains on a token string built by wrapping each turn in that model's own special delimiters, and Llama 3, Mistral, Qwen, and Gemma all use different ones. Get the wrapper wrong and nothing errors; the run just trains on a format the model has never seen.

In [ ]:
# CHAT TEMPLATES - the same messages become a DIFFERENT token string per model.
# In real code you never hand-write these: tokenizer.apply_chat_template(messages,
# tokenize=False) reads the model's Jinja2 template and emits the exact sequence it
# was trained on. Here we simulate four templates to see why the wrapper matters.
msgs = [{"role": "user", "content": "Refund?"}, {"role": "assistant", "content": "7 days."}]

def render(model, m):
    u, a = m[0]["content"], m[1]["content"]
    if model == "llama-3":
        return (f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{u}<|eot_id|>"
                f"<|start_header_id|>assistant<|end_header_id|>\n\n{a}<|eot_id|>")
    if model == "mistral":
        return f"<s>[INST] {u} [/INST] {a}</s>"
    if model == "qwen":
        return f"<|im_start|>user\n{u}<|im_end|>\n<|im_start|>assistant\n{a}<|im_end|>"
    if model == "gemma":
        return f"<start_of_turn>user\n{u}<end_of_turn>\n<start_of_turn>model\n{a}<end_of_turn>"

for mdl in ("llama-3", "mistral", "qwen", "gemma"):
    print(f"  {mdl:9s}: {render(mdl, msgs)!r}")

# The SILENT failure: Alpaca-format text has NONE of Llama-3's special tokens.
alpaca = f"### Instruction:\n{msgs[0]['content']}\n### Response:\n{msgs[1]['content']}"
print("\n  Alpaca text fed to a Llama-3 model (WRONG - no special tokens):")
print("   ", repr(alpaca))
print("  -> trains with no error and no <|eot_id|> stop token = silent garbage.")

# Output:
#   llama-3  : '<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nRefund?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n7 days.<|eot_id|>'
#   mistral  : '<s>[INST] Refund? [/INST] 7 days.</s>'
#   qwen     : '<|im_start|>user\nRefund?<|im_end|>\n<|im_start|>assistant\n7 days.<|im_end|>'
#   gemma    : '<start_of_turn>user\nRefund?<end_of_turn>\n<start_of_turn>model\n7 days.<end_of_turn>'
#
#   Alpaca text fed to a Llama-3 model (WRONG - no special tokens):
#     '### Instruction:\nRefund?\n### Response:\n7 days.'
#   -> trains with no error and no <|eot_id|> stop token = silent garbage.

**Explanation**

A simulation cell: `render` hand-fakes four models' templates so you can literally see the special tokens, whereas real code always calls `tokenizer.apply_chat_template(messages, tokenize=False)`. The four outputs are the same two messages wrapped four different ways, and the final block shows the silent failure - Alpaca-format text that carries none of Llama 3's tokens.

**How the code works - step by step**
- **`msgs`** - one user turn and one assistant turn, reused for every renderer.
- **`render`** - branches per model and returns the exact delimiter string each was pre-trained on: `<|eot_id|>` for Llama 3, `[/INST]` for Mistral, `<|im_end|>` for Qwen, `<end_of_turn>` for Gemma.
- The loop prints all four wrappings side by side so the difference is unmistakable.
- **`alpaca`** - the same content in `### Instruction / ### Response` form, fed to a Llama 3 model with none of Llama's special tokens present - trains with no error and no stop token.

**In one line:** same messages, four token strings, and only the model's own template is safe.

**What you'll see:** The four model rows each printing their distinctly-wrapped token string, then the Alpaca text printed as a wrong example with the note that it trains with no `<|eot_id|>` stop token - silent garbage.

## 3 - Quality filtering - drop garbage before training

Once the data is correctly shaped you filter it before it ever reaches the GPU. A few cheap, deterministic checks catch most junk - responses that are too short or too long, exact duplicates, and obvious low-quality patterns - and removing roughly 20-40% of raw data is a good thing, not a loss.

In [ ]:
# QUALITY FILTER FUNNEL - drop garbage before it poisons the model.
import hashlib
from collections import Counter

class DataFilter:
    def __init__(self, min_w=10, max_w=500):
        self.min_w, self.max_w = min_w, max_w
        self.seen, self.stats = set(), Counter()

    def _resp(self, item):
        for m in reversed(item.get("messages", [])):
            if m["role"] == "assistant":
                return m["content"]
        return ""

    def filter(self, data):
        clean = []
        for item in data:
            self.stats["total"] += 1
            r = self._resp(item)
            wc = len(r.split())
            if wc < self.min_w: self.stats["too_short"] += 1; continue     # length
            if wc > self.max_w: self.stats["too_long"] += 1; continue
            h = hashlib.md5(r.encode()).hexdigest()
            if h in self.seen: self.stats["duplicate"] += 1; continue       # exact dedup
            self.seen.add(h)
            if "I don't know" in r: self.stats["low_quality"] += 1; continue  # heuristic
            uniq = len(set(r.lower().split())) / max(len(r.split()), 1)
            if uniq < 0.3: self.stats["repetitive"] += 1; continue
            self.stats["kept"] += 1; clean.append(item)
        return clean

def mk(u, a): return {"messages": [{"role": "user", "content": u}, {"role": "assistant", "content": a}]}
raw = [
    mk("Refund?", "Full refund within 7 days of purchase, 50% up to day 30, none after that at all."),
    mk("Help", "Ok."),                                                                # too short
    mk("Price?", "I don't know the exact price right now, please check with the support team about it."),  # low quality
    mk("Refund?", "Full refund within 7 days of purchase, 50% up to day 30, none after that at all."),  # duplicate
    mk("Cost?", "The GenAI Complete Course costs 14,999 rupees and includes lifetime access to everything."),
]
f = DataFilter(min_w=10)
clean = f.filter(raw)
print(f"  {len(raw)} raw -> {len(clean)} clean")
print(f"  breakdown: {dict(f.stats)}")

# Output:
#   5 raw -> 2 clean
#   breakdown: {'total': 5, 'kept': 2, 'too_short': 1, 'low_quality': 1, 'duplicate': 1}

**Explanation**

A filtering harness built as a class that runs each example through an ordered funnel of cheap checks and tallies why things were dropped. It's deterministic on purpose: length first, then an MD5-hash exact-dedup, then quality heuristics, so you get a full breakdown of the drop reasons rather than just a survivor count.

**How the code works - step by step**
- **`DataFilter.__init__`** - stores min/max word thresholds plus a `seen` hash set and a `stats` Counter.
- **`_resp`** - pulls the last assistant message from an example (the text being judged).
- **`filter`** - for each item: reject on word count below `min_w` or above `max_w`, reject if its MD5 hash is already in `seen` (exact duplicate), reject on the `"I don't know"` heuristic, reject if unique-word ratio drops below 0.3 (repetitive); everything surviving is counted as `kept`.
- **`mk`** - tiny helper that wraps a user/assistant pair into the `messages` format.
- The `raw` list of 5 is deliberately seeded with one too-short, one low-quality, and one exact-duplicate example.

**In one line:** run cheap checks fastest-first and keep a ledger of every drop.

**What you'll see:** `5 raw -> 2 clean`, followed by the breakdown dict showing total 5, kept 2, and one each of too_short, low_quality, and duplicate.

## 4 - Quality crushes quantity - the AlpaGasus filter

Cheap heuristics catch obvious junk; the subtle cases need judgment. The AlpaGasus method (ICLR 2024) uses an LLM as a judge to score each example 1-5 on accuracy, helpfulness, and relevance, then keeps only the high scorers - filtering 52,000 Alpaca examples down to 9,000 this way beat the full-data model and trained far faster.

In [ ]:
# QUALITY > QUANTITY (AlpaGasus): score each example, keep only the best.
# A real judge is an LLM rating 1-5 on accuracy/helpfulness/relevance. Here we use a
# deterministic stand-in so the demo runs offline; the RULE is what matters.
def judge_score(resp):
    words = resp.split(); n = len(words)
    length_ok = 1.0 if 8 <= n <= 120 else (0.4 if n < 8 else 0.7)
    specific = 1.0 if any(c.isdigit() for c in resp) else 0.6       # concrete facts score higher
    hedge = 0.3 if ("I don't know" in resp or "check the website" in resp.lower()) else 1.0
    return round(5 * (0.4*length_ok + 0.3*specific + 0.3*hedge), 2)  # 1-5 scale

pool = [
    "Full refund within 7 days, 50% from day 8 to 30, none after 30.",
    "We have a refund policy, please check the website for details.",
    "The GenAI course costs 14,999 rupees with lifetime access and GCP credits.",
    "I don't know, sorry.",
    "Live classes run daily at 7 PM IST; recordings are posted within 2 hours.",
]
KEEP = 4.5
scored = [(judge_score(r), r) for r in pool]
kept = [r for s, r in scored if s >= KEEP]
for s, r in sorted(scored, reverse=True):
    print(f"  {s:.2f}  {'KEEP' if s >= KEEP else 'drop'}  {r[:52]}")
print(f"  AlpaGasus rule: keep score >= {KEEP} -> {len(kept)}/{len(pool)} ({round(100*len(kept)/len(pool))}%) survive")

# Output:
#   5.00  KEEP  The GenAI course costs 14,999 rupees with lifetime a
#   5.00  KEEP  Live classes run daily at 7 PM IST; recordings are p
#   5.00  KEEP  Full refund within 7 days, 50% from day 8 to 30, non
#   3.35  drop  We have a refund policy, please check the website fo
#   2.15  drop  I don't know, sorry.
#   AlpaGasus rule: keep score >= 4.5 -> 3/5 (60%) survive

**Explanation**

A scoring cell that stands in for an LLM judge with a deterministic function so the demo runs offline. `judge_score` rewards concrete, complete, non-hedging answers on a 1-5 scale, and a single `KEEP = 4.5` threshold enacts the AlpaGasus rule: keep only the best.

**How the code works - step by step**
- **`judge_score`** - blends three signals: `length_ok` (full credit for 8-120 words), `specific` (higher if the answer contains a digit), and `hedge` (penalized for `"I don't know"` or `"check the website"`), scaled to 1-5.
- **`pool`** - five candidate responses mixing three concrete answers with one vague and one hedging one.
- **`scored`** - pairs each response with its score; **`kept`** filters to score `>= KEEP`.
- The sorted print shows every response tagged KEEP or drop, then the survival rate.

**In one line:** grade each example, keep the A-grades, and let a smaller high-quality set win.

**What you'll see:** The five responses ranked by score - three at 5.00 marked KEEP, the vague one at 3.35 and `"I don't know"` at 2.15 dropped - ending with `keep score >= 4.5 -> 3/5 (60%) survive`.

## Setup - synthetic-data dependencies

The next concept makes a live model call, so this cell installs its two packages. It stays commented out so a Restart & Run All is safe on a machine that already has them; uncomment it on Colab or a fresh environment.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install google-genai python-dotenv -q  # noqa


**Explanation**

A dependency-install cell for the synthetic-data step, deliberately commented so it does nothing unless you opt in.

**How the code works - step by step**
- A commented `pip install google-genai python-dotenv -q` line (with `# noqa`) - `google-genai` is the current Google GenAI SDK, `python-dotenv` lets you load keys from a `.env` file.

**In one line:** opt-in install for the two packages the synthetic-data cell needs.

**What you'll see:** (no output - commented-out install)

## Setup - load the API key

The synthetic-data demo needs a Google API key. This cell reads it from the environment (never hardcode it) and defaults it to empty so the guarded demo below can skip gracefully when no key is set.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

An environment-prep cell that ensures a `GOOGLE_API_KEY` variable exists without overwriting one you've already set.

**How the code works - step by step**
- Imports `os` and calls `os.environ.setdefault("GOOGLE_API_KEY", "")`, which only sets an empty default if the key isn't already present (get yours from aistudio.google.com).
- Keeping it empty by default is what lets the next cell's `if` guard fall through to a friendly message instead of crashing.

**In one line:** make sure the key variable exists, without clobbering a real one.

**What you'll see:** (no output - environment prep)

## 5 - Synthetic data generation - and always filter it

Sometimes you don't have enough data, so the modern fix is to generate it: prompt a strong model with your knowledge base to produce instruction examples. This is mainstream and legitimate (Microsoft's Phi models leaned on it), with one non-negotiable rule - synthetic data is raw data and must go through the same cleaning and quality filter as anything you collected.

> **Needs a Google API key** (set GOOGLE_API_KEY) - the cell is guarded and prints a skip message without one.

In [ ]:
# SYNTHETIC DATA - use a strong model to generate training examples, then FILTER.
# Requires GOOGLE_API_KEY and `pip install google-genai` (see the two cells above).
# Guarded so Restart & Run All is safe without a key; set the key to run it live.
if os.environ.get("GOOGLE_API_KEY"):
    from google import genai
    from google.genai import types

    client = genai.Client()   # reads GOOGLE_API_KEY

    KB = ("Netsetos GenAI course: 14,999 rupees, lifetime access. "
          "Refund: full within 7 days, 50% day 8-30, none after 30. Live classes 7 PM IST daily.")

    def generate_instruction(n: int = 5) -> list[dict]:
        prompt = (f"Generate {n} diverse user questions about this knowledge base plus ideal "
                  f'assistant answers. Return a JSON array of {{"user": "...", "assistant": "..."}}.\n'
                  f"Knowledge:\n{KB}\nJSON:")
        txt = client.models.generate_content(
            model="gemini-2.5-flash", contents=prompt,
            config=types.GenerateContentConfig(temperature=0.7)).text
        import json, re
        txt = re.sub(r"```(json)?", "", txt).strip()
        pairs = json.loads(txt)
        return [{"messages": [{"role": "system", "content": "You are a Netsetos support assistant."},
                              {"role": "user", "content": p["user"]},
                              {"role": "assistant", "content": p["assistant"]}]} for p in pairs]

    examples = generate_instruction(n=5)
    print(f"Generated {len(examples)} synthetic instruction examples. First:")
    print(" ", examples[0]["messages"][1]["content"])
    print("  ALWAYS filter synthetic data (dedup + quality score) before training on it.")
else:
    print("Set GOOGLE_API_KEY (and pip install google-genai) to run the live synthetic-data demo.")

# Output: (varies by run - synthetic generation is non-deterministic)
# Generated 5 synthetic instruction examples. First:
#   How long do I have to get a full refund on the GenAI course?
#   ALWAYS filter synthetic data (dedup + quality score) before training on it.

**Explanation**

A live generation cell wrapped in an `if os.environ.get("GOOGLE_API_KEY")` guard so Restart & Run All is safe with no key. It uses the current `from google import genai` SDK with `gemini-2.5-flash`, prompts the model with a knowledge base, and parses the returned JSON array back into the step-1 messages format.

**How the code works - step by step**
- **Guard** - only runs the live path if a key is present; otherwise prints an instruction to set the key.
- **`client = genai.Client()`** - the current SDK entry point, reads `GOOGLE_API_KEY` automatically (the retired `google.generativeai` + `gemini-2.0-flash` path is replaced).
- **`KB`** - a short Netsetos knowledge base the questions must be grounded in.
- **`generate_instruction`** - builds a prompt asking for `n` diverse question/answer pairs as a JSON array, calls `generate_content` with temperature 0.7, strips any code fences with a regex, `json.loads` the text, and reshapes each pair into a system/user/assistant `messages` record.
- The final print restates the load-bearing rule: always dedup and quality-score synthetic output before training.

**In one line:** generate from a knowledge base, parse to messages, then filter it like any raw data.

**What you'll see:** With a key: `Generated 5 synthetic instruction examples.` plus the first generated question and the reminder to always filter. Output varies run to run since generation is non-deterministic; without a key, the skip message prints instead.

## 6 - The cleaning pipeline - cheapest filters first

Beyond quality lies cleanliness, and a production pipeline runs its filters ordered by cost: the free ones first, the LLM-based ones last. This cell chains four stages - exact dedup (free), near-duplicate detection, a contamination check against a held-out benchmark, and PII replacement - which is exactly the order that keeps you from hand-sorting garbage you could have screened cheaply.

In [ ]:
# THE CLEANING PIPELINE - run the cheapest filters first.
import hashlib, re

def shingles(text, k=3):
    w = text.lower().split()
    return set(tuple(w[i:i+k]) for i in range(max(len(w)-k+1, 1)))

def jaccard(a, b):
    A, B = shingles(a), shingles(b)
    return len(A & B) / max(len(A | B), 1)

# a tiny held-out benchmark we must NEVER train on (contamination = benchmark leakage)
BENCHMARK = ["what is the capital of france", "who wrote pride and prejudice"]
def contaminated(text, bench, n=4):
    tg = shingles(text, n)
    return any(shingles(b, n) & tg for b in bench)   # shared 4-gram = leaked eval question

PII = [(re.compile(r"\b\d{4}\s?\d{4}\s?\d{4}\b"), "<AADHAAR>"),
       (re.compile(r"\b[\w.]+@[\w.]+\.\w+\b"), "<EMAIL>"),
       (re.compile(r"\b(?:\+91[- ]?)?[6-9]\d{9}\b"), "<PHONE>")]
def scrub(text):
    for pat, tag in PII:
        text = pat.sub(tag, text)   # REPLACE with a placeholder (not delete) to keep structure
    return text

docs = [
    "You can get a full refund within 7 days of purchase. Contact rahul@netsetos.com or call 9876543210 for help.",
    "You can get a full refund within 7 days of purchase. Reach rahul@netsetos.com or call 9876543210 for help.",   # near-duplicate
    "My Aadhaar number is 1234 5678 9012 and I would like to request a refund on my order today.",
    "Quick quiz to warm up: what is the capital of france? Great, now back to your refund question.",              # contaminated
]
seen, exact = set(), []
for d in docs:                                   # 1) exact dedup (free)
    h = hashlib.md5(d.encode()).hexdigest()
    if h not in seen: seen.add(h); exact.append(d)
near = []
for d in exact:                                  # 2) near-dup (cheap) - real code uses MinHash/LSH
    if all(jaccard(d, k) <= 0.6 for k in near): # via datasketch with a tuned Jaccard threshold
        near.append(d)
clean = [d for d in near if not contaminated(d, BENCHMARK)]   # 3) drop benchmark leakage
final = [scrub(d) for d in clean]                # 4) PII placeholder-replace (before any LLM step)
print(f"  {len(docs)} docs -> exact {len(exact)} -> near-dedup {len(near)} -> "
      f"decontaminated {len(clean)} -> PII-scrubbed:")
for d in final:
    print("   ", d)

# Output:
#   4 docs -> exact 4 -> near-dedup 3 -> decontaminated 2 -> PII-scrubbed:
#     You can get a full refund within 7 days of purchase. Contact <EMAIL> or call <PHONE> for help.
#     My Aadhaar number is <AADHAAR> and I would like to request a refund on my order today.

**Explanation**

A four-stage cleaning pipeline that runs each free/cheap filter before any expensive one. It shows a hash catching exact dupes, shingle-based Jaccard catching reformulations a hash misses, an n-gram check dropping benchmark leakage, and regex PII replacement that swaps in typed placeholders rather than deleting text.

**How the code works - step by step**
- **`shingles` / `jaccard`** - turn text into sets of word 3-grams and measure overlap, the cheap near-duplicate signal (real code uses MinHash/LSH via datasketch).
- **`contaminated`** - flags any doc sharing a 4-gram with the held-out `BENCHMARK`, so eval questions never leak into training.
- **`PII` + `scrub`** - regexes for Aadhaar, email, and phone that replace matches with `<AADHAAR>`/`<EMAIL>`/`<PHONE>` placeholders (keeping the sentence trainable).
- **The pipeline** - stage 1 exact-dedup by MD5, stage 2 near-dedup by Jaccard `<= 0.6`, stage 3 drop contaminated docs, stage 4 scrub PII on the survivors.
- `docs` is seeded with one near-duplicate and one benchmark-leak so each stage visibly fires.

**In one line:** hash, then shingle, then decontaminate, then redact - cheapest first.

**What you'll see:** `4 docs -> exact 4 -> near-dedup 3 -> decontaminated 2 -> PII-scrubbed:` followed by the two surviving docs with their email/phone and Aadhaar replaced by placeholders.

## 7 - Splits, versioning & tools

Clean data still needs to be split and tracked. Split 80/10/10 for small sets and stratify by task type so every category appears in each split, keep at least 50-100 validation examples or the loss curve is noise, then version it (DVC git-tracks large files via tiny metafiles; `push_to_hub()` shares it with a dataset card).

In [ ]:
# STRATIFIED SPLIT - keep every task type proportional across train/val/test.
from collections import Counter

data = ([{"task": "refund", "id": i} for i in range(10)] +
        [{"task": "pricing", "id": i} for i in range(10)] +
        [{"task": "schedule", "id": i} for i in range(10)])

def stratified_split(rows, ratios=(0.8, 0.1, 0.1)):
    by = {}
    for r in rows:
        by.setdefault(r["task"], []).append(r)
    train, val, test = [], [], []
    for task, items in sorted(by.items()):      # per-task, so proportions are preserved
        n = len(items)
        n_tr, n_va = round(n*ratios[0]), round(n*ratios[1])
        train += items[:n_tr]; val += items[n_tr:n_tr+n_va]; test += items[n_tr+n_va:]
    return train, val, test

tr, va, te = stratified_split(data)
print(f"  {len(data)} examples -> train {len(tr)} / val {len(va)} / test {len(te)}")
print(f"  train task mix: {dict(Counter(r['task'] for r in tr))}  (each task kept proportional)")

# Output:
#   30 examples -> train 24 / val 3 / test 3
#   train task mix: {'pricing': 8, 'refund': 8, 'schedule': 8}  (each task kept proportional)

**Explanation**

A splitting harness that demonstrates why stratification matters. It groups rows by task, splits each group 80/10/10 independently, and concatenates - so a task-sorted dataset can't dump a whole category into one split the way a plain slice would.

**How the code works - step by step**
- **`data`** - 30 rows, 10 each of three tasks (refund/pricing/schedule), deliberately sorted by task.
- **`stratified_split`** - buckets rows into `by[task]`, then for each task computes `n_tr`/`n_va` from the ratios and slices that group into train/val/test before concatenating.
- Because the split happens per task, each task keeps its proportion in every split - the alternative plain slice would put whole tasks into one bucket.
- The prints show the total split sizes and confirm the train set's task mix is even.

**In one line:** split each task separately so val/test actually mirror the data.

**What you'll see:** `30 examples -> train 24 / val 3 / test 3`, then `train task mix: {'pricing': 8, 'refund': 8, 'schedule': 8}` showing all three tasks kept proportional.

## 8 - India data preparation

Indian-language data adds three twists. **Tokenizer fertility**: a generic tokenizer may split Hindi into 4-8 tokens per word (Telugu even more), truncating examples and multiplying inference cost, so you pick the base model by fertility first. **Normalization**: always apply Unicode NFC before tokenizing. And **Hinglish** code-mixing has no standard spelling - the AI4Bharat ecosystem supplies open, DPDP-friendly data.

In [ ]:
# INDIA - tokenizer fertility decides viability, and NFC-normalize before tokenizing.
import unicodedata

# tokens-per-word by (model, language) - illustrative from AI4Bharat / Sarvam reporting.
FERT = {"llama-3": {"english": 1.3, "hindi": 6.0, "telugu": 11.0},
        "sarvam-1": {"english": 1.5, "hindi": 1.8, "telugu": 2.0}}
CTX = 4096
for lang in ("english", "hindi", "telugu"):
    la, sa = FERT["llama-3"][lang], FERT["sarvam-1"][lang]
    print(f"  {lang:8s}: llama-3 ~{la:4.1f} tok/word ({int(CTX/la):4d} words fit a {CTX}-ctx) | "
          f"sarvam-1 ~{sa:3.1f} tok/word ({int(CTX/sa):4d} words)")
print("  -> On Hindi/Telugu, Sarvam-1's Indic tokenizer fits 3-5x more words per context than Llama-3;")
print("     on English the two are comparable. Choose the base model by fertility on YOUR language.")

decomposed = "café"   # combining acute accent - looks identical to an e-acute
nfc = unicodedata.normalize("NFC", decomposed)
print(f"  NFC: {len(decomposed)} code points -> {len(nfc)} (combining marks collapse to one).")

# Output:
#   english : llama-3 ~ 1.3 tok/word (3150 words fit a 4096-ctx) | sarvam-1 ~1.5 tok/word (2730 words)
#   hindi   : llama-3 ~ 6.0 tok/word ( 682 words fit a 4096-ctx) | sarvam-1 ~1.8 tok/word (2275 words)
#   telugu  : llama-3 ~11.0 tok/word ( 372 words fit a 4096-ctx) | sarvam-1 ~2.0 tok/word (2048 words)
#   -> On Hindi/Telugu, Sarvam-1's Indic tokenizer fits 3-5x more words per context than Llama-3;
#      on English the two are comparable. Choose the base model by fertility on YOUR language.
#   NFC: 5 code points -> 4 (combining marks collapse to one).

**Explanation**

A measurement cell, not a model call: it uses an illustrative fertility table to compute how many words of each language fit a fixed context, then demonstrates NFC normalization collapsing a combining accent. The gap between Llama 3 and Sarvam-1 on Hindi/Telugu is the whole point.

**How the code works - step by step**
- **`FERT`** - a lookup of tokens-per-word by (model, language) drawn from AI4Bharat / Sarvam reporting; `CTX = 4096` is the context budget.
- The loop divides `CTX` by each fertility to print how many words fit for Llama 3 vs Sarvam-1 across English, Hindi, and Telugu - showing Sarvam-1 fits several times more Indic content per context.
- **`unicodedata.normalize("NFC", ...)`** - collapses a decomposed `cafe` + combining acute (5 code points) to the single-codepoint form (4), which is why identical-looking text must be normalized before hashing or tokenizing.

**In one line:** choose the base model by fertility on your language, and NFC-normalize before tokenizing.

**What you'll see:** Three fertility rows (english/hindi/telugu) contrasting Llama 3's and Sarvam-1's tokens-per-word and words-that-fit, the note that Sarvam-1 fits 3-5x more Indic words per context, and `NFC: 5 code points -> 4` showing the combining mark collapse.

Across eight steps you took raw text all the way to a training-ready dataset: shaped it into instruction/conversation/preference JSONL, wrapped it with the model's own chat template (the one silent-failure step), filtered on quality over quantity with heuristics and an AlpaGasus-style judge, generated and cleaned synthetic data cheapest-filters-first, then split, versioned, and Indic-tuned it for fertility and NFC. The through-line: data prep is roughly 80% of fine-tuning, and quality - of both content and FORMAT - beats quantity every time. The JSONL you produce here is exactly what Lesson 9.3 (Vertex managed tuning), 9.4 (open-source Axolotl/Unsloth/TRL), and 9.5 (the LoRA/QLoRA training loop) consume next.